In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 데이터 불러오기 (파일명을 확인하세요)
train_df = pd.read_csv('security_log_train.csv')
test_df = pd.read_csv('security_log_test.csv')
submission = pd.read_csv('sample_submission.csv')

text_col = 'full_log'
target_col = 'level'

# 2. 마스킹 함수 정의
def mask_log_text(text):
    if not isinstance(text, str): return text
    text = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '<IP>', text)
    text = re.sub(r'\b\d{4}[-/]\d{2}[-/]\d{2}\b', '<DATE>', text)
    text = re.sub(r'\b\d{2}:\d{2}:\d{2}(?:\.\d+)?\b', '<TIME>', text)
    text = re.sub(r'\b0x[0-9a-fA-F]+\b', '<HEX>', text)
    text = re.sub(r'\b\d+\b', '<NUM>', text)
    return text

print("데이터 전처리 중...")
train_df[text_col] = train_df[text_col].apply(mask_log_text)
test_df[text_col] = test_df[text_col].apply(mask_log_text)

# 3. 벡터화
print("벡터화 중...")
vectorizer = TfidfVectorizer(max_features=10000, analyzer='word', ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(train_df[text_col]).astype(np.float32)
X_test_vec = vectorizer.transform(test_df[text_col]).astype(np.float32)
y_train = train_df[target_col]

# 4. 데이터 분할
X_tr, X_val, y_tr, y_val = train_test_split(X_train_vec, y_train, test_size=0.2, random_state=42, stratify=y_train)

# 5. 모델 학습 (앙상블을 위해 2개 모델 선택)
print("앙상블 모델 학습 시작...")
model1 = LogisticRegression(C=0.05, max_iter=1000)
model2 = LGBMClassifier(n_estimators=300, device='gpu', reg_alpha=0.3, reg_lambda=0.5)

model1.fit(X_tr, y_tr)
model2.fit(X_tr, y_tr)

# 6. 확률 예측 및 결합
prob1 = model1.predict_proba(X_val)
prob2 = model2.predict_proba(X_val)
ensemble_probs = (prob1 + prob2) / 2
ensemble_preds = ensemble_probs.argmax(axis=1)

print(f"🎉 앙상블 검증 Macro F1: {f1_score(y_val, ensemble_preds, average='macro'):.4f}")

# 7. 확률 예측 및 결합
prob1 = model1.predict_proba(X_test_vec)
prob2 = model2.predict_proba(X_test_vec)
final_probs = (prob1 + prob2) / 2 # 앙상블 확률값

# 8. Threshold 후처리 적용
THRESHOLD = 0.7  # (이전 실험에서 가장 좋았던 값으로 설정하세요)
max_probs = final_probs.max(axis=1) # 앙상블된 확률값 중 가장 높은 값
final_preds = final_probs.argmax(axis=1)

# 확률이 낮은 데이터는 '7'번으로 예측 (자신 없는 예측 방지)
final_preds[max_probs < THRESHOLD] = 7

# 제출 파일 저장
submission[target_col] = final_preds
submission.to_csv(f'final_ensemble_thresh_{THRESHOLD}.csv', index=False)

데이터 전처리 중...
벡터화 중...
앙상블 모델 학습 시작...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 331149
[LightGBM] [Info] Number of data points in the train set: 378377, number of used features: 9302
[LightGBM] [Info] Using GPU Device: NVIDIA RTX A4000, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 15 dense feature groups (5.77 MB) transferred to GPU in 0.005689 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score -0.347699
[LightGBM] [Info] Start training from score -1.272329
[LightGBM] [Info] Start training from score -10.541061
[LightGBM] [Info] Start training from score -4.738037
[LightGBM] [Info] Start training from score -10.764205
[LightGBM] [Info] Start training from score -5.362091
[LightGBM] [Info] Start training from score -11.051887
[LightGBM] [Warning] No further splits with positive gai

c:\Users\user\anaconda3\envs\ml_env01\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


🎉 앙상블 검증 Macro F1: 0.8994


c:\Users\user\anaconda3\envs\ml_env01\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
